In [11]:

import sys
import numpy as np
import pandas as pd
from scipy.optimize import minimize
from scipy.stats import chi2 as chi2d

XLSX = "3.xlsx"
ALPHA = np.pi / 4

df = pd.read_excel(XLSX, sheet_name="Dati")
df = df.dropna(subset=["lambda", "n'", "err_lambda_max", "sigma_n"])
D = df.to_numpy(dtype="float64")
th0, x, y = D[:, 0], D[:, 2], D[:, 11]
ex, ey = D[:, 3] / np.sqrt(3), D[:, 13]
delta, sig_d = D[:, 10], D[:, 12]


def sell(l, A, B):
    return np.sqrt(1 + A * l**2 / (l**2 - B**2))


def d_sell(l, A, B, dl=1e-6):
    return (sell(l + dl / 2, A, B) - sell(l - dl / 2, A, B)) / dl


def retta(l, A, B):
    return A + B * l


def d_retta(l, A, B):
    return B * np.ones_like(l)


def fit(f, fd, p0, x, y, ex, ey):
    """chi2 con varianza efficace ricalcolata ad ogni passo (come TGraphErrors di ROOT)."""
    def c2(p):
        s = np.sqrt(ey**2 + (fd(x, *p) * ex) ** 2)
        return np.sum(((y - f(x, *p)) / s) ** 2)
    r = minimize(c2, p0, method="Nelder-Mead",
                 options=dict(xatol=1e-12, fatol=1e-12, maxiter=20000, maxfev=20000))
    r = minimize(c2, r.x, method="Nelder-Mead",
                 options=dict(xatol=1e-13, fatol=1e-13, maxiter=20000, maxfev=20000))
    return r.x, r.fun


def res_norm(p, mask):
    s = np.sqrt(ey[mask] ** 2 + (d_sell(x[mask], *p) * ex[mask]) ** 2)
    return (y[mask] - sell(x[mask], *p)) / s


#     fit e chi2
print(" Fit (A, B, chi2/ndf, p) ")
casi = [[], [671, 690], [620, 656], [620, 656, 671, 690]]
par = {}
for tol in casi:
    k = ~np.isin(x, tol)
    p, c = fit(sell, d_sell, [1.27, 100.0], x[k], y[k], ex[k], ey[k])
    nd = k.sum() - 2
    par[tuple(tol)] = p
    print(f"senza {str(tol):22s} A={p[0]:.4f}  B={p[1]:.1f}  chi2={c:.2f}/{nd}  p={chi2d.sf(c, nd):.3f}")
pl, cl = fit(retta, d_retta, [1.54, 0.0], x, y, ex, ey)
print(f"retta (tutti)                 A={pl[0]:.4f}  B={pl[1]:.3e}  chi2={cl:.2f}/12  p={chi2d.sf(cl, 12):.1e}")

# errore relativo su delta
print(" Errore relativo su delta dei punti esclusi (fit senza quei punti) ")
for tol in ([671, 690], [620, 656]):
    idx = np.isin(x, tol)
    n_f = sell(x[idx], *par[tuple(tol)])
    d_f = 2 * np.arcsin(n_f * np.sin(ALPHA / 2)) - ALPHA
    err = np.abs(d_f - delta[idx]) / d_f * 100
    primi = np.degrees(np.abs(d_f - delta[idx])) * 60
    for l, e, m in zip(x[idx], err, primi):
        print(f"lambda={l:.0f} nm: err.rel.={e:.2f} %   (= {m:.1f} primi = {m:.1f} tacche)")

delta, alpha = D[:, 10], np.pi/4

dn_dd = np.cos((alpha + delta)/2) / (2*np.sin(alpha/2))   # dn/d(delta)

print(f"delta: {delta.min():.3f} - {delta.max():.3f} rad")
print(f"dn/ddelta: {dn_dd.min():.4f} - {dn_dd.max():.4f} rad^-1")


 Fit (A, B, chi2/ndf, p) 
senza []                     A=1.2679  B=95.0  chi2=19.87/12  p=0.070
senza [671, 690]             A=1.2714  B=91.4  chi2=4.35/10  p=0.930
senza [620, 656]             A=1.2661  B=96.6  chi2=5.71/10  p=0.839
senza [620, 656, 671, 690]   A=1.2692  B=93.5  chi2=1.24/8  p=0.996
retta (tutti)                 A=1.5475  B=-5.046e-05  chi2=55.25/12  p=1.6e-07
 Errore relativo su delta dei punti esclusi (fit senza quei punti) 
lambda=671 nm: err.rel.=0.21 %   (= 3.3 primi = 3.3 tacche)
lambda=690 nm: err.rel.=0.25 %   (= 3.8 primi = 3.8 tacche)
lambda=620 nm: err.rel.=0.19 %   (= 2.9 primi = 2.9 tacche)
lambda=656 nm: err.rel.=0.22 %   (= 3.4 primi = 3.4 tacche)
delta: 0.450 - 0.463 rad
dn/ddelta: 1.0600 - 1.0651 rad^-1
